In [0]:
from pyspark.sql import *
from pyspark.sql.functions import udf



data = [
    Row(Product_ID=1, name="Paracetamol", category="FeverMedicine",ListPrice=100),
    Row(Product_ID=2, name="ORS", category="StomachUpset",ListPrice=150),
    Row(Product_ID=3, name="Dettol", category="Disinfectant",ListPrice=200)
]

df = spark.createDataFrame(data)
display(df)

table_name='bronze.orders'
print(f"Table name is {table_name}")

#Creating a function
def clean_amount(amt):
    if amt is None:
        return 0
    return amt * 1.18

clean_amount_udf = udf(clean_amount)
df = df.withColumn("ListPrice", clean_amount_udf(df["ListPrice"]))
display(df)

tables=["Orders","Customers","Products"]
for t in tables:
    print(f"Processing{t}")
    




In [0]:
from pyspark.sql.types import StructType,StructField,StringType,LongType
product_schema=StructType(
    [
    StructField("Product_ID",LongType()),
    StructField("name",StringType()),
    StructField("category",StringType()),
    StructField("ListPrice",LongType())
                ]
    )

    

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS my_catalog;
CREATE SCHEMA IF NOT EXISTS my_catalog.bronze;

In [0]:
from pyspark.sql.functions import col,when
df=spark.read.format("csv").option("header","true").schema(product_schema).load("/Workspace/Users/nsomia109@gmail.com/DataBricks/Product.csv")
display(df)

df.write.mode("overwrite").format("delta").saveAsTable("my_catalog.bronze.Product")
df=spark.read.table("my_catalog.bronze.Product")
df_clean=df.withColumn("Status",when(col("ListPrice")>100,"Expensive").otherwise("Cheap")).filter(col("ListPrice")>100).drop("ListPrice")
df_clean.display()

In [0]:
dbutils.fs.ls("/Workspace/Users/nsomia109@gmail.com/DataBricks")
spark.sql("create schema if not exists my_catalog.silver")
df_clean.write.mode("overwrite").format("delta").saveAsTable("my_catalog.silver.Product")
df=spark.read.table("my_catalog.silver.Product")
display(df)
dbutils.widgets.text("env","Dev")
env=dbutils.widgets.get("env")
print(env)

In [0]:
Data=[(1,"John","USA"),(2,"Mary","Canada"),(3,"Steve","India"),(4,"Lara","Australia")]
Columns=["Order_id","Name","Country"]
df=spark.createDataFrame(Data,Columns)
df.show()


In [0]:
from pyspark.sql.functions import concat, col
from delta.tables import DeltaTable

#df_concat = df.withColumn("concatenated", concat(col("Order_id"), col("Name")))
#df_concat.show()
df_agg = df.withColumn("Amount", col("Order_id") * 10)
df_agg.show()
from pyspark.sql.functions import sum,count,avg,max
df_agg.groupBy("Country").agg(sum("Amount").alias("TOtalSum"),count("Amount"),avg("Amount"),max("Amount")).show()

data=[(1,3,"TV",100,"2022-01-01"),(2,3,"cosmetic",200,"2021-01-01"),(3,2,"Grocery",300,"2023-01-01"),(4,2,"Medicines",400,"2024-01-01")]
Columns=["Sales_id","Order_id","Product","Cost","Order_date"]
df2=spark.createDataFrame(data,Columns)
df2.show()

df.join(df2,df.Order_id==df2.Order_id,"inner").show()
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number,desc,lag
w=Window.partitionBy("Order_id").orderBy(desc("Order_date"))
df2.withColumn("rn",row_number().over(w)).filter(col("rn")==1).show()
df2.withColumn("prev_amnt",lag("Cost",1).over(w)).show()

df2.na.fill({"Cost": 0, "Product": "Unknown"}).show()
from pyspark.sql.functions import lit
df2.withColumn("prev_amnt",lag("Cost",1).over(w)).na.fill({"prev_amnt": 0}).show()
spark.sql("create schema if not exists my_catalog.silver")
df2.write.mode("overwrite").format("delta").saveAsTable("my_catalog.silver.Product")
# Example: update a Delta table using the DeltaTable API
# delta_table = DeltaTable.forName(spark, "my_catalog.silver.Product")
# delta_table.update(condition=col("Status") == "Cheap", set={"Status": lit("Affordable")})

In [0]:
# Example: update a Delta table using the DeltaTable API
from pyspark.sql.functions import *
from delta.tables import DeltaTable
delta_table = DeltaTable.forName(spark, "my_catalog.silver.Product")
delta_table.update(condition=col("Sales_id") == 1, set={"Cost": lit(None)})

#spark.sql("select * from my_catalog.silver.Product").show()

df1=spark.table("my_catalog.silver.Product").drop("Product_ID","name","category","Status")
#df1.show();
df1.write.mode("overwrite").format("delta").option("overwriteSchema","true").saveAsTable("my_catalog.silver.Product")
df2=spark.table("my_catalog.silver.Product")
df2.show()

#dropna
df2.dropna(subset=["Cost","Order_date"],how="all").show()

#widgets
dbutils.widgets.text("table","silver.Product","Table")
tbl=dbutils.widgets.get("table")
print(tbl)
df2=spark.table(f"my_catalog.{tbl}")
df2.show()
dbutils.widgets.dropdown("table2","silver.Product",["silver.Product","Orders"])
tbl1=dbutils.widgets.get("table2")
print(tbl1)
dbutils.widgets.removeAll()
df2.printSchema()
#Versioning
#spark.sql("DESCRIBE HISTORY my_catalog.silver.Product limit 2").show(truncate=False)

from delta.tables import DeltaTable
delta_table = DeltaTable.forName(spark, "my_catalog.silver.Product")
delta_table.history(2).show(truncate=False)

spark.read.option("versionAsOf", 150).table("my_catalog.silver.Product").show()

spark.sql("RESTORE TABLE my_catalog.silver.Product TO VERSION AS OF 149")





In [0]:
#Merge
df2=spark.table("my_catalog.silver.Product")
data=[(2,3,"cosmetic","800","2026-02-01"),(6,2,"Medicines","1000","2025-01-01"),(5,2,"Laptop","900","2026-01-28")]
Columns=["Sales_id","Order_id","Product","Cost","Order_date"]
df3=spark.createDataFrame(data,Columns)
df3.show()
df3.createOrReplaceTempView("Updates")
df2.show()
#spark.sql("""
#Merge into my_catalog.silver.Product as t
#using Updates as s
#on t.Sales_id=s.Sales_id
#when matched then update set *
#when not matched then insert *
#""")
df2.show()

#Merge in pyspark
from delta.tables import DeltaTable
delta_table=DeltaTable.forName(spark,"my_catalog.silver.Product")
delta_table.alias("t").merge(df3.alias("s"),"t.Sales_id=s.Sales_id").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
df2.show()
#Example: delete a Delta table using the DeltaTable API
from delta.tables import DeltaTable
delta_table = DeltaTable.forName(spark, "my_catalog.silver.Product")
delta_table.delete("Sales_id=1")
df2.show()
#merging selected fields
from pyspark.sql.functions import when, col, lit
df3=df3.withColumns({"Cost":when(col("Sales_id")==2,lit(2000)).otherwise(col("Cost")),"Product":when(col("Sales_id")==2,lit("Healthcare")).otherwise(col("Product"))
})

df3.show()
df2.show()

delta_table.alias("t").merge(df3.alias("s"),"t.sales_id=s.sales_id").whenMatchedUpdate(set={"Cost":"s.Cost","Product":"s.Product"}).whenNotMatchedInsert(values={"Sales_id":"s.Sales_id","Order_id":"s.Order_id","Cost":"s.Cost","Product":"s.Product","Order_date":"s.Order_date"}) \
.whenNotMatchedBySourceDelete().execute()
df2.show()
#adding new ros
new_data=[(7,2,"Phone","500","2025-01-01"),(8,3,"Tablet","300","2025-01-01")]
new_rows=spark.createDataFrame(new_data,df2.columns)
df2=df2.unionByName(new_rows)
df2.show()
spark.sql("DESCRIBE TABLE my_catalog.silver.Product").show(truncate=False)



In [0]:
dbutils.fs.ls("/")